# SBA 7(a) Loan Risk — Gradient Boosting Challenger

This notebook develops a Gradient Boosting classification model
to predict recorded charge-offs within five calendar years.

The challenger is compared against the Logistic Regression baseline
using the same chronological evaluation design.

## Chronological periods

- Train: FY2010–2015
- Tune: FY2016
- Calibrate: FY2017
- Validate: FY2018
- Test: FY2019

## Primary evaluation metrics

- Average Precision (PR-AUC)
- Recall at Top 10% review capacity

## Objective

Determine whether Gradient Boosting provides meaningful improvements
over the simpler Logistic Regression baseline.

> **Reproducibility note:** This notebook was originally executed from the project `notebooks/` directory using Python 3.12.8 and scikit-learn 1.5.1. The source CSV (`../data/exports/sba_model_export.csv`) is not distributed in GitHub because it exceeds the normal file-size limit. Generate it using the documented PostgreSQL export workflow before executing cells. Saved outputs below are historical results; editing documentation does not constitute a new execution.


In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
DATA_PATH = Path("../data/exports/sba_model_export.csv")

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Missing modeling CSV: {DATA_PATH}. Generate it via the PostgreSQL export and run from notebooks/."
    )

In [2]:
df = pd.read_csv(
    DATA_PATH,
    parse_dates=["approval_date"],
    dtype={
        "sba_lender_id": "string",
        "lender_key": "string"
    }
)

print("Dataset shape:", df.shape)

assert df.shape == (478953, 25)

Dataset shape: (478953, 25)


> **Historical experiment (15 features):** The initial Gradient Boosting model below includes loan term and shows unusually strong performance. Those figures are *not* the final leakage-reviewed results. This notebook investigates the issue, drops loan term and its flag, and then tunes revised 13-feature candidates.


In [3]:
numeric_features = [
    "gross_approval",
    "guarantee_pct",
    "term_months_model",
    "initial_interest_rate_model",
    "jobs_supported"
]

categorical_features = [
    "business_age_group",
    "naics_sector",
    "project_state",
    "processing_method",
    "collateral_indicator",
    "revolver_status",
    "interest_rate_type",
    "business_type"
]

flag_features = [
    "invalid_term_flag",
    "invalid_interest_rate_flag"
]

feature_columns = (
    numeric_features
    + categorical_features
    + flag_features
)

target_column = "chargeoff_within_5yr"

print("Candidate predictors:", len(feature_columns))

Candidate predictors: 15


In [4]:
train_df = df.loc[
    df["modeling_period"] == "Train"
].copy()

tune_df = df.loc[
    df["modeling_period"] == "Tune"
].copy()

X_train = train_df[feature_columns].copy()
y_train = train_df[target_column].copy()

X_tune = tune_df[feature_columns].copy()
y_tune = tune_df[target_column].copy()

print("Training:", X_train.shape, y_train.shape)
print("Tuning:", X_tune.shape, y_tune.shape)

assert X_train.shape == (266226, 15)
assert X_tune.shape == (56785, 15)

print("\nChronological split validation passed!")

Training: (266226, 15) (266226,)
Tuning: (56785, 15) (56785,)

Chronological split validation passed!


In [5]:
# Count distinct categories using training data only
category_counts = (
    X_train[categorical_features]
    .nunique(dropna=False)
    .sort_values(ascending=False)
)

print("Categorical feature cardinality:")
print(category_counts)

# Check categorical features against maximum bin limit
max_bins = 255

high_cardinality_features = category_counts[
    category_counts > max_bins
]

print("\nFeatures exceeding 255 categories:")
print(high_cardinality_features)

if high_cardinality_features.empty:
    print("\nAll categorical features are within the limit.")

Categorical feature cardinality:
project_state           56
naics_sector            21
processing_method       21
business_age_group       8
business_type            4
collateral_indicator     2
revolver_status          2
interest_rate_type       2
dtype: int64

Features exceeding 255 categories:
Series([], dtype: int64)

All categorical features are within the limit.


In [6]:
# Create independent feature copies
X_train_gb = X_train.copy()
X_tune_gb = X_tune.copy()

# Store categorical values learned from training data
category_mappings = {}

for feature in categorical_features:

    # Learn categories from training data only
    train_categories = pd.Index(
        X_train_gb[feature].dropna().unique()
    )

    category_mappings[feature] = train_categories

    # Apply identical categories to training and tuning data
    categorical_dtype = pd.CategoricalDtype(
        categories=train_categories
    )

    X_train_gb[feature] = X_train_gb[feature].astype(
        categorical_dtype
    )

    X_tune_gb[feature] = X_tune_gb[feature].astype(
        categorical_dtype
    )

print("Training categorical dtypes:")
print(X_train_gb[categorical_features].dtypes)

print("\nTuning categorical dtypes:")
print(X_tune_gb[categorical_features].dtypes)

Training categorical dtypes:
business_age_group      category
naics_sector            category
project_state           category
processing_method       category
collateral_indicator    category
revolver_status         category
interest_rate_type      category
business_type           category
dtype: object

Tuning categorical dtypes:
business_age_group      category
naics_sector            category
project_state           category
processing_method       category
collateral_indicator    category
revolver_status         category
interest_rate_type      category
business_type           category
dtype: object


In [7]:
# Confirm all categorical columns were converted
assert all(
    isinstance(X_train_gb[col].dtype, pd.CategoricalDtype)
    for col in categorical_features
)

assert all(
    isinstance(X_tune_gb[col].dtype, pd.CategoricalDtype)
    for col in categorical_features
)

# Confirm that training and tuning use identical category sets
assert all(
    X_train_gb[col].cat.categories.equals(
        X_tune_gb[col].cat.categories
    )
    for col in categorical_features
)

# Check whether tuning contains unseen categories
print("Unseen categories in FY2016:")

for feature in categorical_features:
    unseen_count = (
        X_tune[feature].notna()
        & X_tune_gb[feature].isna()
    ).sum()

    print(f"{feature}: {unseen_count}")

print("\nCategorical preparation validation passed!")

Unseen categories in FY2016:
business_age_group: 0
naics_sector: 0
project_state: 1
processing_method: 0
collateral_indicator: 0
revolver_status: 0
interest_rate_type: 0
business_type: 0

Categorical preparation validation passed!


In [8]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb_model = HistGradientBoostingClassifier(
    learning_rate=0.1,
    max_iter=100,
    max_leaf_nodes=15,
    min_samples_leaf=50,
    l2_regularization=1.0,
    categorical_features="from_dtype",
    early_stopping=False,
    random_state=RANDOM_STATE
)

print(gb_model)

HistGradientBoostingClassifier(categorical_features='from_dtype',
                               early_stopping=False, l2_regularization=1.0,
                               max_leaf_nodes=15, min_samples_leaf=50,
                               random_state=42)


In [9]:
import time

# Record training start time
start_time = time.perf_counter()

# Train Gradient Boosting using historical training loans
gb_model.fit(X_train_gb, y_train)

# Calculate training duration
training_seconds = time.perf_counter() - start_time

print("Gradient Boosting training completed.")
print(f"Training time: {training_seconds:.2f} seconds")
print("Boosting iterations completed:", gb_model.n_iter_)
print("Number of input features:", gb_model.n_features_in_)

Gradient Boosting training completed.
Training time: 2.65 seconds
Boosting iterations completed: 100
Number of input features: 15


In [10]:
# Generate FY2016 charge-off probabilities
gb_tune_probabilities = gb_model.predict_proba(X_tune_gb)[:, 1]

# Inspect the probability distribution
print("Number of predictions:", len(gb_tune_probabilities))

print(
    "Minimum probability:",
    gb_tune_probabilities.min()
)

print(
    "Maximum probability:",
    gb_tune_probabilities.max()
)

print(
    "Average predicted probability:",
    gb_tune_probabilities.mean()
)

# Validate probability output
assert len(gb_tune_probabilities) == len(y_tune)

assert np.isfinite(gb_tune_probabilities).all()

assert (
    (gb_tune_probabilities >= 0)
    & (gb_tune_probabilities <= 1)
).all()

print("\nGradient Boosting probability validation passed!")

Number of predictions: 56785
Minimum probability: 7.434422646825481e-05
Maximum probability: 0.8663403618718842
Average predicted probability: 0.03707294902412385

Gradient Boosting probability validation passed!


In [11]:
# Actual charge-off incidence in FY2016
gb_tune_prevalence = y_tune.mean()

# Calculate Average Precision
gb_tune_ap = average_precision_score(
    y_tune,
    gb_tune_probabilities
)

# Calculate AP lift over prevalence
gb_ap_lift = gb_tune_ap / gb_tune_prevalence

# Calculate ROC-AUC
gb_tune_roc_auc = roc_auc_score(
    y_tune,
    gb_tune_probabilities
)

print(f"FY2016 event prevalence: {gb_tune_prevalence:.4%}")
print(f"Gradient Boosting Average Precision: {gb_tune_ap:.4f}")
print(f"Average Precision lift: {gb_ap_lift:.2f}x")
print(f"Gradient Boosting ROC-AUC: {gb_tune_roc_auc:.4f}")

FY2016 event prevalence: 3.8707%
Gradient Boosting Average Precision: 0.5672
Average Precision lift: 14.65x
Gradient Boosting ROC-AUC: 0.9688


In [12]:
# Number of loans the lender can review
gb_review_count = int(np.ceil(len(y_tune) * 0.10))

# Rank loans by predicted risk, highest first
gb_top_indices = np.argsort(
    -gb_tune_probabilities,
    kind="stable"
)[:gb_review_count]

# Identify actual outcomes in the highest-risk group
gb_top_actual = y_tune.to_numpy()[gb_top_indices]

# Count actual charge-offs
gb_captured_chargeoffs = int(gb_top_actual.sum())
gb_total_chargeoffs = int(y_tune.sum())

# Calculate operational metrics
gb_recall_10 = gb_captured_chargeoffs / gb_total_chargeoffs
gb_precision_10 = gb_captured_chargeoffs / gb_review_count
gb_precision_lift_10 = gb_precision_10 / gb_tune_prevalence

print("Total FY2016 loans:", len(y_tune))
print("Loans selected for review:", gb_review_count)
print("Total actual charge-offs:", gb_total_chargeoffs)
print("Charge-offs captured:", gb_captured_chargeoffs)

print(f"\nRecall at Top 10%: {gb_recall_10:.2%}")
print(f"Precision at Top 10%: {gb_precision_10:.2%}")
print(f"Precision lift: {gb_precision_lift_10:.2f}x")

# Validate calculations
assert gb_review_count == 5679
assert gb_total_chargeoffs == 2198
assert 0 <= gb_recall_10 <= 1
assert 0 <= gb_precision_10 <= 1

print("\nGradient Boosting operational metrics validated!")

Total FY2016 loans: 56785
Loans selected for review: 5679
Total actual charge-offs: 2198
Charge-offs captured: 1942

Recall at Top 10%: 88.35%
Precision at Top 10%: 34.20%
Precision lift: 8.83x

Gradient Boosting operational metrics validated!


In [13]:
from sklearn.inspection import permutation_importance

# Measure feature importance on FY2016 tuning data
permutation_results = permutation_importance(
    gb_model,
    X_tune_gb,
    y_tune,
    scoring="average_precision",
    n_repeats=2,
    random_state=RANDOM_STATE,
    n_jobs=1
)

# Build a readable feature importance table
importance_df = pd.DataFrame({
    "Feature": X_tune_gb.columns,
    "AP_Decrease": permutation_results.importances_mean,
    "Std_Dev": permutation_results.importances_std
})

importance_df = importance_df.sort_values(
    "AP_Decrease",
    ascending=False
).reset_index(drop=True)

display(importance_df.round(4))

,Feature,AP_Decrease,Std_Dev
0,term_months_model,0.5209,0.0008
1,initial_interest_rate_model,0.1365,0.0080
2,processing_method,0.0618,0.0011
3,project_state,0.0510,0.0018
4,revolver_status,0.0222,0.0039
5,gross_approval,0.0213,0.0048
6,interest_rate_type,0.0203,0.0025
7,guarantee_pct,0.0196,0.0007
8,naics_sector,0.0067,0.0001
9,jobs_supported,0.0033,0.0007


In [14]:
# Create an analysis-only copy of training records
term_analysis = train_df[
    ["term_months_model", "chargeoff_within_5yr"]
].copy()

# Group loan terms into meaningful duration bands
term_analysis["term_group"] = pd.cut(
    term_analysis["term_months_model"],
    bins=[0, 60, 120, 180, 240, 300, np.inf],
    labels=[
        "1–60 months",
        "61–120 months",
        "121–180 months",
        "181–240 months",
        "241–300 months",
        "Over 300 months"
    ],
    include_lowest=True
)

# Calculate loan counts and charge-off incidence
term_summary = (
    term_analysis
    .groupby("term_group", observed=True)
    .agg(
        loan_count=("chargeoff_within_5yr", "size"),
        chargeoffs=("chargeoff_within_5yr", "sum"),
        chargeoff_rate=("chargeoff_within_5yr", "mean")
    )
    .reset_index()
)

term_summary["chargeoff_rate_pct"] = (
    term_summary["chargeoff_rate"] * 100
)

display(
    term_summary[
        [
            "term_group",
            "loan_count",
            "chargeoffs",
            "chargeoff_rate_pct"
        ]
    ].round(2)
)

print(
    "Missing or invalid loan terms:",
    term_analysis["term_months_model"].isna().sum()
)

,term_group,loan_count,chargeoffs,chargeoff_rate_pct
0,1–60 months,58101,5232,9.01
1,61–120 months,149807,4288,2.86
2,121–180 months,14155,101,0.71
3,181–240 months,10207,99,0.97
4,241–300 months,30992,158,0.51
5,Over 300 months,2808,0,0.00


Missing or invalid loan terms: 156


In [15]:
# Calculate historical charge-off rates by exact loan term
exact_term_summary = (
    train_df
    .groupby("term_months_model", dropna=False)
    .agg(
        loan_count=("chargeoff_within_5yr", "size"),
        chargeoffs=("chargeoff_within_5yr", "sum"),
        chargeoff_rate=("chargeoff_within_5yr", "mean")
    )
    .reset_index()
)

# Convert rates to percentages
exact_term_summary["chargeoff_rate_pct"] = (
    exact_term_summary["chargeoff_rate"] * 100
)

# Identify the 20 most common loan terms
most_common_terms = (
    exact_term_summary
    .sort_values("loan_count", ascending=False)
    .head(20)
)

display(
    most_common_terms[
        [
            "term_months_model",
            "loan_count",
            "chargeoffs",
            "chargeoff_rate_pct"
        ]
    ].round(2)
)

,term_months_model,loan_count,chargeoffs,chargeoff_rate_pct
83,84.0,82315,96,0.12
119,120.0,41784,34,0.08
59,60.0,27871,141,0.51
299,300.0,27352,0,0.00
239,240.0,6833,0,0.00
179,180.0,4571,2,0.04
35,36.0,4541,130,2.86
47,48.0,3473,124,3.57
71,72.0,3176,108,3.40
11,12.0,3079,42,1.36


In [16]:
# Use training data only for this diagnostic
term_pattern_df = train_df[
    ["term_months_model", "chargeoff_within_5yr"]
].copy()

# Exclude missing or invalid terms from this comparison
term_pattern_df = term_pattern_df.dropna(
    subset=["term_months_model"]
)

# Identify terms that are exact multiples of 12 months
term_pattern_df["term_pattern"] = np.where(
    term_pattern_df["term_months_model"] % 12 == 0,
    "Multiple of 12 months",
    "Other term length"
)

# Compare charge-off incidence between groups
term_pattern_summary = (
    term_pattern_df
    .groupby("term_pattern")
    .agg(
        loan_count=("chargeoff_within_5yr", "size"),
        chargeoffs=("chargeoff_within_5yr", "sum"),
        chargeoff_rate=("chargeoff_within_5yr", "mean")
    )
    .reset_index()
)

term_pattern_summary["chargeoff_rate_pct"] = (
    term_pattern_summary["chargeoff_rate"] * 100
)

display(
    term_pattern_summary[
        [
            "term_pattern",
            "loan_count",
            "chargeoffs",
            "chargeoff_rate_pct"
        ]
    ].round(2)
)

,term_pattern,loan_count,chargeoffs,chargeoff_rate_pct
0,Multiple of 12 months,215093,880,0.41
1,Other term length,50977,8998,17.65


In [17]:
from sklearn.base import clone

# Exclude loan term for a controlled sensitivity experiment
features_without_term = [
    feature for feature in feature_columns
    if feature != "term_months_model"
]

X_train_no_term = X_train_gb[features_without_term].copy()
X_tune_no_term = X_tune_gb[features_without_term].copy()

# Create a fresh Gradient Boosting model
gb_no_term = clone(gb_model)

# Train using FY2010–2015 only
gb_no_term.fit(X_train_no_term, y_train)

# Predict FY2016 charge-off probabilities
gb_no_term_probabilities = gb_no_term.predict_proba(
    X_tune_no_term
)[:, 1]

# Calculate Average Precision
gb_no_term_ap = average_precision_score(
    y_tune,
    gb_no_term_probabilities
)

# Select the highest-risk 10%
review_count = int(np.ceil(len(y_tune) * 0.10))

top_indices = np.argsort(
    -gb_no_term_probabilities,
    kind="stable"
)[:review_count]

captured = int(y_tune.to_numpy()[top_indices].sum())

# Calculate operational metrics
gb_no_term_recall = captured / y_tune.sum()
gb_no_term_precision = captured / review_count

print("Features used:", len(features_without_term))
print(f"Average Precision: {gb_no_term_ap:.4f}")
print(f"Recall@10%: {gb_no_term_recall:.2%}")
print(f"Precision@10%: {gb_no_term_precision:.2%}")
print("Charge-offs captured:", captured)

Features used: 14
Average Precision: 0.0903
Recall@10%: 26.75%
Precision@10%: 10.35%
Charge-offs captured: 588


In [18]:
import joblib

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load the previously selected Logistic Regression pipeline
baseline_path = Path("../models/logistic_regression_baseline.joblib")
if not baseline_path.is_file():
    raise FileNotFoundError(
        f"Missing baseline model: {baseline_path}. Run Notebook 02 first from notebooks/."
    )
saved_logistic = joblib.load(baseline_path)

# Remove loan term from numerical predictors
numeric_without_term = [
    feature for feature in numeric_features
    if feature != "term_months_model"
]

# Recreate the same preprocessing with 14 predictors
preprocessor_no_term = ColumnTransformer(
    transformers=[
        (
            "num",
            clone(saved_logistic.named_steps[
                "preprocessor"
            ].named_transformers_["num"]),
            numeric_without_term
        ),
        (
            "cat",
            clone(saved_logistic.named_steps[
                "preprocessor"
            ].named_transformers_["cat"]),
            categorical_features
        ),
        ("flags", "passthrough", flag_features)
    ],
    remainder="drop"
)

# Build an unfitted Logistic Regression pipeline
logistic_no_term = Pipeline(
    steps=[
        ("preprocessor", preprocessor_no_term),
        (
            "classifier",
            clone(saved_logistic.named_steps["classifier"])
        )
    ]
)

# Train on FY2010–2015
logistic_no_term.fit(
    X_train[features_without_term],
    y_train
)

# Generate FY2016 predictions
lr_no_term_probabilities = logistic_no_term.predict_proba(
    X_tune[features_without_term]
)[:, 1]

# Calculate performance
lr_no_term_ap = average_precision_score(
    y_tune, lr_no_term_probabilities
)

top_indices = np.argsort(
    -lr_no_term_probabilities,
    kind="stable"
)[:review_count]

lr_captured = int(y_tune.to_numpy()[top_indices].sum())

lr_no_term_recall = lr_captured / y_tune.sum()
lr_no_term_precision = lr_captured / review_count

print("Features used:", len(features_without_term))
print(f"Average Precision: {lr_no_term_ap:.4f}")
print(f"Recall@10%: {lr_no_term_recall:.2%}")
print(f"Precision@10%: {lr_no_term_precision:.2%}")
print("Charge-offs captured:", lr_captured)

Features used: 14
Average Precision: 0.0805
Recall@10%: 23.70%
Precision@10%: 9.17%
Charge-offs captured: 521


In [19]:
# Measure feature importance for Gradient Boosting without loan term
importance_no_term = permutation_importance(
    gb_no_term,
    X_tune_no_term,
    y_tune,
    scoring="average_precision",
    n_repeats=3,
    random_state=RANDOM_STATE,
    n_jobs=1
)

# Build an importance summary
importance_no_term_df = pd.DataFrame({
    "Feature": X_tune_no_term.columns,
    "AP_Decrease": importance_no_term.importances_mean,
    "Std_Dev": importance_no_term.importances_std
})

importance_no_term_df = (
    importance_no_term_df
    .sort_values("AP_Decrease", ascending=False)
    .reset_index(drop=True)
)

display(importance_no_term_df.round(4))

,Feature,AP_Decrease,Std_Dev
0,initial_interest_rate_model,0.0221,0.0007
1,gross_approval,0.0153,0.0020
2,processing_method,0.0125,0.0012
3,naics_sector,0.0075,0.0004
4,project_state,0.0072,0.0012
5,business_age_group,0.0066,0.0017
6,guarantee_pct,0.0053,0.0003
7,jobs_supported,0.0042,0.0014
8,collateral_indicator,0.0026,0.0004
9,interest_rate_type,0.0014,0.0003


## Leakage review and revised 13-feature candidates

The preceding 15-feature and 14-feature experiments are retained as diagnostics. The final feature revision excludes both `term_months_model` and `invalid_term_flag`. These revised candidates are evaluated further in subsequent validation notebooks; selection on FY2016 alone is not the final model decision.


In [20]:
# Exclude loan term and its derived data-quality flag
excluded_risk_features = [
    "term_months_model",
    "invalid_term_flag"
]

revised_features = [
    feature
    for feature in feature_columns
    if feature not in excluded_risk_features
]

print("Original predictors:", len(feature_columns))
print("Revised predictors:", len(revised_features))
print("\nRevised feature list:")

for feature in revised_features:
    print(feature)

assert len(revised_features) == 13

print("\nRevised feature selection validated!")

Original predictors: 15
Revised predictors: 13

Revised feature list:
gross_approval
guarantee_pct
initial_interest_rate_model
jobs_supported
business_age_group
naics_sector
project_state
processing_method
collateral_indicator
revolver_status
interest_rate_type
business_type
invalid_interest_rate_flag

Revised feature selection validated!


In [21]:
# Select the revised 13 predictors
X_train_revised = X_train_gb[revised_features].copy()
X_tune_revised = X_tune_gb[revised_features].copy()

# Verify dataset dimensions
print("Training:", X_train_revised.shape)
print("Tuning:", X_tune_revised.shape)

assert X_train_revised.shape == (266226, 13)
assert X_tune_revised.shape == (56785, 13)

print("\nRevised dataset validation passed!")

Training: (266226, 13)
Tuning: (56785, 13)

Revised dataset validation passed!


In [22]:
from sklearn.base import clone

# Create a fresh model with the original configuration
gb_revised = clone(gb_model)

# Train using FY2010–2015 only
gb_revised.fit(X_train_revised, y_train)

# Generate FY2016 predictions
gb_revised_probabilities = gb_revised.predict_proba(
    X_tune_revised
)[:, 1]

# Calculate Average Precision
gb_revised_ap = average_precision_score(
    y_tune,
    gb_revised_probabilities
)

# Calculate ROC-AUC
gb_revised_roc_auc = roc_auc_score(
    y_tune,
    gb_revised_probabilities
)

# Select the highest-risk 10%
review_count = int(np.ceil(len(y_tune) * 0.10))

top_indices = np.argsort(
    -gb_revised_probabilities,
    kind="stable"
)[:review_count]

# Calculate operational metrics
gb_revised_captured = int(
    y_tune.to_numpy()[top_indices].sum()
)

gb_revised_recall = gb_revised_captured / y_tune.sum()

gb_revised_precision = gb_revised_captured / review_count

# Display results
print("Model: Revised Gradient Boosting")
print("Predictors:", gb_revised.n_features_in_)
print(f"Average Precision: {gb_revised_ap:.4f}")
print(f"ROC-AUC: {gb_revised_roc_auc:.4f}")
print(f"Recall@10%: {gb_revised_recall:.2%}")
print(f"Precision@10%: {gb_revised_precision:.2%}")
print("Charge-offs captured:", gb_revised_captured)

# Validate predictions and model structure
assert gb_revised.n_features_in_ == 13
assert len(gb_revised_probabilities) == len(y_tune)
assert np.isfinite(gb_revised_probabilities).all()
assert ((gb_revised_probabilities >= 0) & (gb_revised_probabilities <= 1)).all()

print("\nRevised Gradient Boosting validation passed!")

Model: Revised Gradient Boosting
Predictors: 13
Average Precision: 0.0911
ROC-AUC: 0.7345
Recall@10%: 27.02%
Precision@10%: 10.46%
Charge-offs captured: 594

Revised Gradient Boosting validation passed!


In [23]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Define the revised numerical features
revised_numeric_features = [
    "gross_approval",
    "guarantee_pct",
    "initial_interest_rate_model",
    "jobs_supported"
]

# Only the interest-rate quality flag remains
revised_flag_features = [
    "invalid_interest_rate_flag"
]

# Retrieve existing preprocessing components
previous_preprocessor = logistic_no_term.named_steps[
    "preprocessor"
]

# Create a fresh preprocessing structure
revised_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            clone(previous_preprocessor.named_transformers_["num"]),
            revised_numeric_features
        ),
        (
            "cat",
            clone(previous_preprocessor.named_transformers_["cat"]),
            categorical_features
        ),
        (
            "flags",
            "passthrough",
            revised_flag_features
        )
    ],
    remainder="drop"
)

# Create the revised Logistic Regression pipeline
lr_revised = Pipeline(
    steps=[
        ("preprocessor", revised_preprocessor),
        (
            "classifier",
            clone(logistic_no_term.named_steps["classifier"])
        )
    ]
)

print("Revised Logistic Regression pipeline created.")
print("Predictors:", len(revised_features))
print("Regularization C:", lr_revised.named_steps["classifier"].C)

Revised Logistic Regression pipeline created.
Predictors: 13
Regularization C: 1.0


In [24]:
# Prepare the same 13 predictor columns
X_train_lr_revised = X_train[revised_features].copy()
X_tune_lr_revised = X_tune[revised_features].copy()

# Train using FY2010–2015
lr_revised.fit(
    X_train_lr_revised,
    y_train
)

# Generate FY2016 probabilities
lr_revised_probabilities = lr_revised.predict_proba(
    X_tune_lr_revised
)[:, 1]

# Calculate ranking metrics
lr_revised_ap = average_precision_score(
    y_tune,
    lr_revised_probabilities
)

lr_revised_roc_auc = roc_auc_score(
    y_tune,
    lr_revised_probabilities
)

# Select highest-risk 10%
lr_top_indices = np.argsort(
    -lr_revised_probabilities,
    kind="stable"
)[:review_count]

lr_revised_captured = int(
    y_tune.to_numpy()[lr_top_indices].sum()
)

lr_revised_recall = lr_revised_captured / y_tune.sum()
lr_revised_precision = lr_revised_captured / review_count

# Display results
print("Model: Revised Logistic Regression")
print("Predictors:", len(revised_features))
print(f"Average Precision: {lr_revised_ap:.4f}")
print(f"ROC-AUC: {lr_revised_roc_auc:.4f}")
print(f"Recall@10%: {lr_revised_recall:.2%}")
print(f"Precision@10%: {lr_revised_precision:.2%}")
print("Charge-offs captured:", lr_revised_captured)

# Validate model outputs
assert len(lr_revised_probabilities) == len(y_tune)
assert np.isfinite(lr_revised_probabilities).all()
assert ((lr_revised_probabilities >= 0) &
        (lr_revised_probabilities <= 1)).all()

print("\nRevised Logistic Regression validation passed!")

Model: Revised Logistic Regression
Predictors: 13
Average Precision: 0.0805
ROC-AUC: 0.7186
Recall@10%: 23.70%
Precision@10%: 9.17%
Charge-offs captured: 521

Revised Logistic Regression validation passed!


In [25]:
# Candidate regularization strengths
revised_c_values = [0.1, 1.0, 10.0]

# Store models and evaluation results
revised_lr_tuning_results = []
revised_lr_models = {}

print("Revised Logistic Regression tuning configurations:")

for c in revised_c_values:
    print(f"C = {c}")

print("\nTotal configurations:", len(revised_c_values))

Revised Logistic Regression tuning configurations:
C = 0.1
C = 1.0
C = 10.0

Total configurations: 3


In [26]:
from sklearn.base import clone

# Use the same review capacity for every configuration
review_count = int(np.ceil(len(y_tune) * 0.10))

revised_lr_tuning_results = []
revised_lr_models = {}

for c in revised_c_values:

    # Create a fresh, unfitted pipeline
    model = clone(lr_revised)

    # Set the regularization strength
    model.set_params(classifier__C=c)

    # Train using the revised 13 predictors
    model.fit(X_train_lr_revised, y_train)

    # Predict FY2016 charge-off probabilities
    probabilities = model.predict_proba(
        X_tune_lr_revised
    )[:, 1]

    # Calculate ranking metrics
    ap = average_precision_score(
        y_tune,
        probabilities
    )

    roc_auc = roc_auc_score(
        y_tune,
        probabilities
    )

    # Identify highest-risk 10% of loans
    top_indices = np.argsort(
        -probabilities,
        kind="stable"
    )[:review_count]

    captured = int(
        y_tune.to_numpy()[top_indices].sum()
    )

    # Calculate operational metrics
    recall_10 = captured / y_tune.sum()
    precision_10 = captured / review_count

    # Check optimization iterations
    iterations = int(
        model.named_steps["classifier"].n_iter_[0]
    )

    # Store the evaluation results
    revised_lr_tuning_results.append({
        "C": c,
        "Average_Precision": ap,
        "ROC_AUC": roc_auc,
        "Recall_at_10": recall_10,
        "Precision_at_10": precision_10,
        "Captured_Chargeoffs": captured,
        "Iterations": iterations
    })

    # Store the fitted pipeline
    revised_lr_models[c] = model

    print(f"Completed C = {c}")

# Create the comparison DataFrame
revised_lr_tuning_df = pd.DataFrame(
    revised_lr_tuning_results
)

print("\nRevised Logistic Regression Tuning Results:")

display(revised_lr_tuning_df.round(4))

Completed C = 0.1
Completed C = 1.0
Completed C = 10.0

Revised Logistic Regression Tuning Results:


,C,Average_Precision,ROC_AUC,Recall_at_10,Precision_at_10,Captured_Chargeoffs,Iterations
0,0.1,0.0804,0.7185,0.2348,0.0909,516,29
1,1.0,0.0805,0.7186,0.2370,0.0917,521,34
2,10.0,0.0803,0.7183,0.2366,0.0916,520,31


In [27]:
# Select the configuration with the highest FY2016 AP
best_lr_result = revised_lr_tuning_df.loc[
    revised_lr_tuning_df["Average_Precision"].idxmax()
]

best_lr_c = float(best_lr_result["C"])

# Retrieve the previously fitted model
selected_lr_revised = revised_lr_models[best_lr_c]

print("Selected C:", best_lr_c)
print(f"Average Precision: {best_lr_result['Average_Precision']:.4f}")
print(f"Recall@10%: {best_lr_result['Recall_at_10']:.2%}")
print(f"Precision@10%: {best_lr_result['Precision_at_10']:.2%}")

# Validate model configuration
assert best_lr_c == 1.0
assert selected_lr_revised.named_steps["classifier"].C == best_lr_c

print("\nRevised Logistic Regression selection validated!")

Selected C: 1.0
Average Precision: 0.0805
Recall@10%: 23.70%
Precision@10%: 9.17%

Revised Logistic Regression selection validated!


In [28]:
# Gradient Boosting hyperparameter configurations
gb_configurations = [
    {
        "name": "A - Original reference",
        "learning_rate": 0.10,
        "max_iter": 100,
        "max_leaf_nodes": 15
    },
    {
        "name": "B - Slower learning",
        "learning_rate": 0.05,
        "max_iter": 200,
        "max_leaf_nodes": 15
    },
    {
        "name": "C - Simpler trees",
        "learning_rate": 0.10,
        "max_iter": 100,
        "max_leaf_nodes": 7
    },
    {
        "name": "D - More complex trees",
        "learning_rate": 0.10,
        "max_iter": 100,
        "max_leaf_nodes": 31
    },
    {
        "name": "E - Slower, more complex",
        "learning_rate": 0.05,
        "max_iter": 200,
        "max_leaf_nodes": 31
    }
]

# Prepare containers for future tuning results
gb_tuning_results = []
gb_tuned_models = {}

print("Gradient Boosting configurations:")

for config in gb_configurations:
    print(
        f"{config['name']} | "
        f"learning_rate={config['learning_rate']} | "
        f"max_iter={config['max_iter']} | "
        f"max_leaf_nodes={config['max_leaf_nodes']}"
    )

print("\nTotal configurations:", len(gb_configurations))

Gradient Boosting configurations:
A - Original reference | learning_rate=0.1 | max_iter=100 | max_leaf_nodes=15
B - Slower learning | learning_rate=0.05 | max_iter=200 | max_leaf_nodes=15
C - Simpler trees | learning_rate=0.1 | max_iter=100 | max_leaf_nodes=7
D - More complex trees | learning_rate=0.1 | max_iter=100 | max_leaf_nodes=31
E - Slower, more complex | learning_rate=0.05 | max_iter=200 | max_leaf_nodes=31

Total configurations: 5


In [29]:
from sklearn.base import clone
import time

# Review capacity: highest-risk 10% of FY2016 loans
review_count = int(np.ceil(len(y_tune) * 0.10))

gb_tuning_results = []
gb_tuned_models = {}

for config in gb_configurations:

    # Create a fresh model
    model = clone(gb_revised)

    # Apply this configuration's hyperparameters
    model.set_params(
        learning_rate=config["learning_rate"],
        max_iter=config["max_iter"],
        max_leaf_nodes=config["max_leaf_nodes"]
    )

    # Fit using FY2010–2015 only
    start_time = time.perf_counter()

    model.fit(X_train_revised, y_train)

    training_seconds = time.perf_counter() - start_time

    # Predict FY2016 charge-off probabilities
    probabilities = model.predict_proba(
        X_tune_revised
    )[:, 1]

    # Validate probability outputs
    assert len(probabilities) == len(y_tune)
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()

    # Calculate ranking metrics
    ap = average_precision_score(
        y_tune,
        probabilities
    )

    roc_auc = roc_auc_score(
        y_tune,
        probabilities
    )

    # Select exactly the highest-risk 10%
    top_indices = np.argsort(
        -probabilities,
        kind="stable"
    )[:review_count]

    captured = int(
        y_tune.to_numpy()[top_indices].sum()
    )

    recall_10 = captured / y_tune.sum()
    precision_10 = captured / review_count

    # Record results
    gb_tuning_results.append({
        "Configuration": config["name"],
        "Learning_Rate": config["learning_rate"],
        "Max_Iter": config["max_iter"],
        "Max_Leaf_Nodes": config["max_leaf_nodes"],
        "Average_Precision": ap,
        "ROC_AUC": roc_auc,
        "Recall_at_10": recall_10,
        "Precision_at_10": precision_10,
        "Captured_Chargeoffs": captured,
        "Training_Seconds": training_seconds
    })

    # Save the fitted configuration in memory
    gb_tuned_models[config["name"]] = model

    print(f"Completed: {config['name']}")

# Compare all configurations
gb_tuning_df = pd.DataFrame(gb_tuning_results)

print("\nGradient Boosting Tuning Results:")

display(
    gb_tuning_df.round(4)
)

Completed: A - Original reference
Completed: B - Slower learning
Completed: C - Simpler trees
Completed: D - More complex trees
Completed: E - Slower, more complex

Gradient Boosting Tuning Results:


,Configuration,Learning_Rate,Max_Iter,Max_Leaf_Nodes,Average_Precision,ROC_AUC,Recall_at_10,Precision_at_10,Captured_Chargeoffs,Training_Seconds
0,A - Original reference,0.10,100,15,0.0911,0.7345,0.2702,0.1046,594,1.8017
1,B - Slower learning,0.05,200,15,0.0921,0.7358,0.2739,0.1060,602,2.8370
2,C - Simpler trees,0.10,100,7,0.0885,0.7321,0.2648,0.1025,582,1.6901
3,D - More complex trees,0.10,100,31,0.0904,0.7311,0.2712,0.1049,596,2.1343
4,"E - Slower, more complex",0.05,200,31,0.0923,0.7336,0.2734,0.1058,601,3.5390


In [30]:
# Select the configuration with the highest FY2016 Average Precision
best_gb_result = gb_tuning_df.loc[
    gb_tuning_df["Average_Precision"].idxmax()
]

best_gb_name = best_gb_result["Configuration"]

# Retrieve the already-fitted model
selected_gb_revised = gb_tuned_models[best_gb_name]

print("Selected configuration:", best_gb_name)
print(f"Average Precision: {best_gb_result['Average_Precision']:.4f}")
print(f"Recall@10%: {best_gb_result['Recall_at_10']:.2%}")
print(f"Precision@10%: {best_gb_result['Precision_at_10']:.2%}")
print("Charge-offs captured:", int(best_gb_result["Captured_Chargeoffs"]))

# Verify selected hyperparameters
assert best_gb_name == "E - Slower, more complex"
assert selected_gb_revised.learning_rate == 0.05
assert selected_gb_revised.max_iter == 200
assert selected_gb_revised.max_leaf_nodes == 31
assert selected_gb_revised.n_features_in_ == 13

print("\nGradient Boosting selection validated!")

Selected configuration: E - Slower, more complex
Average Precision: 0.0923
Recall@10%: 27.34%
Precision@10%: 10.58%
Charge-offs captured: 601

Gradient Boosting selection validated!


In [31]:
import joblib
from pathlib import Path

# Create the model output directory
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Define versioned model paths
lr_model_path = (
    MODEL_DIR / "logistic_regression_13_features.joblib"
)

gb_model_path = (
    MODEL_DIR / "gradient_boosting_13_features.joblib"
)

# Save both fitted models
joblib.dump(selected_lr_revised, lr_model_path)
joblib.dump(selected_gb_revised, gb_model_path)

# Confirm that both files exist
print("Logistic Regression saved:", lr_model_path.exists())
print("Gradient Boosting saved:", gb_model_path.exists())

print("\nSaved model files:")
print(lr_model_path.name)
print(gb_model_path.name)

assert lr_model_path.exists()
assert gb_model_path.exists()

print("\nBoth revised models saved successfully!")

Logistic Regression saved: True
Gradient Boosting saved: True

Saved model files:
logistic_regression_13_features.joblib
gradient_boosting_13_features.joblib

Both revised models saved successfully!
